# Rhema Care Flow — Colab Audit Lab
Laboratório Colab para clonar, auditar e validar build do repo. Fonte de verdade: `main` + PR pequeno + Sentinel/TMR. Não cole tokens reais.


In [ ]:
REPO_URL="https://github.com/JoaoRG-lab/rhema-care-flow.git"
REPO_DIR="rhema-care-flow"
BRANCH="main"


In [ ]:
import os, subprocess, pathlib, json
from datetime import datetime

def run(cmd, cwd=None, check=False):
    print(f"\n$ {cmd}")
    p=subprocess.run(cmd,shell=True,cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    print(p.stdout)
    if check and p.returncode: raise RuntimeError(cmd)
    return p.returncode,p.stdout


In [ ]:
if not os.path.exists(REPO_DIR):
    run(f"git clone {REPO_URL} {REPO_DIR}", check=True)
else:
    run("git fetch --all --prune", cwd=REPO_DIR, check=True)
run(f"git checkout {BRANCH}", cwd=REPO_DIR, check=True)
run("git pull --ff-only", cwd=REPO_DIR)
run("git status --short --branch", cwd=REPO_DIR)
run("git log -1 --oneline", cwd=REPO_DIR)


In [ ]:
run("node -v")
run("npm -v")
run("npm install --legacy-peer-deps", cwd=REPO_DIR, check=True)


In [ ]:
env = pathlib.Path(REPO_DIR)/".env.example"
txt = env.read_text(encoding="utf-8") if env.exists() else ""
checks = {
  "has_VITE_SUPABASE_URL": "VITE_SUPABASE_URL" in txt,
  "has_VITE_SUPABASE_PUBLISHABLE_KEY": "VITE_SUPABASE_PUBLISHABLE_KEY" in txt,
  "deprecated_VITE_SUPABASE_ANON_KEY": "VITE_SUPABASE_ANON_KEY" in txt,
  "documents_PERPLEXITY_API_KEY": "PERPLEXITY_API_KEY" in txt,
  "documents_GEMINI_API_KEY": "GEMINI_API_KEY" in txt,
}
print(json.dumps(checks, indent=2, ensure_ascii=False))


In [ ]:
tsc_code, _ = run("npx tsc --noEmit", cwd=REPO_DIR)
lint_code, _ = run("npm run lint --if-present", cwd=REPO_DIR)
build_code, _ = run("npm run build", cwd=REPO_DIR)
print(json.dumps({"tsc":tsc_code,"lint":lint_code,"build":build_code}, indent=2))


In [ ]:
critical = [
 "src/integrations/supabase/client.ts",
 "supabase/functions/ai-assistant/index.ts",
 "src/components/ai/AISiteAgentWidget2.tsx",
 "src/components/ai/AIIntegrationPanel.tsx",
 "src/router.tsx",
 "vercel.json",
 ".github/workflows/tmr-deploy.yml",
 ".github/workflows/audit-sentinel.yml",
]
for f in critical:
    print(f"{f}: {'OK' if (pathlib.Path(REPO_DIR)/f).exists() else 'MISSING'}")


In [ ]:
def read(rel):
    p=pathlib.Path(REPO_DIR)/rel
    return p.read_text(encoding="utf-8") if p.exists() else ""
edge=read("supabase/functions/ai-assistant/index.ts")
panel=read("src/components/ai/AIIntegrationPanel.tsx")
widget=read("src/components/ai/AISiteAgentWidget2.tsx")
contract = {
 "edge_returns_reply": "reply:" in edge,
 "edge_returns_answer": "answer" in edge,
 "panel_reads_reply": "data?.reply" in panel or "data.reply" in panel,
 "panel_sends_agent": "agent:" in panel,
 "widget_site_publico": "site_publico" in widget,
 "edge_cors_reumatismos": "reumatismos.com" in edge,
 "edge_rate_limit": "RATE_LIMIT" in edge or "rateLimitMap" in edge,
}
print(json.dumps(contract, indent=2, ensure_ascii=False))


In [ ]:
report = f'''# Rhema Care Flow — Colab Audit Report
Generated: {datetime.utcnow().isoformat()}Z

Branch: {BRANCH}

Results:
- TypeScript: {tsc_code}
- Lint: {lint_code}
- Build: {build_code}

Interpretation:
- Build 0 = can open/merge small PR after GitHub Sentinel/TMR.
- Nonzero build = inspect first build error before PR.
- Never commit secrets.
'''
pathlib.Path(REPO_DIR, "audit_report.md").write_text(report, encoding="utf-8")
print(report)
